# **Desafio Prático Final: Plataforma "Vagas Tech"**

## **1. Setup do Ambiente e Conexão**

### **1.1 Instalando o .NET SDK na máquina do Colab**

In [ ]:
print("Instalando o repositório de pacotes da Microsoft...")
!wget <https://packages.microsoft.com/config/ubuntu/22.04/packages-microsoft-prod.deb> -O packages-microsoft-prod.deb -q
!dpkg -i packages-microsoft-prod.deb -q
!rm packages-microsoft-prod.deb

print("Instalando o .NET 8 SDK... Por favor, aguarde.")
!apt-get update -y -q > /dev/null
!apt-get install -y dotnet-sdk-8.0 -q > /dev/null

print("\nVerificando a instalação do .NET...")
!dotnet --version

Instalando o repositório de pacotes da Microsoft...
/bin/bash: line 1: https://packages.microsoft.com/config/ubuntu/22.04/packages-microsoft-prod.deb: No such file or directory
dpkg: error: cannot access archive 'packages-microsoft-prod.deb': No such file or directory
rm: cannot remove 'packages-microsoft-prod.deb': No such file or directory
Instalando o .NET 8 SDK... Por favor, aguarde.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)

Verificando a instalação do .NET...
=8.0.131


### **1.2 Criando o projeto e instalando pacotes de SQL**

In [ ]:
print("Criando o novo projeto de console C# 'VagasTechApp'...")
# Cria o projeto na pasta 'VagasTechApp'
!dotnet new console -n VagasTechApp

print("\nInstalando a biblioteca Microsoft.Data.Sqlite para execução de SQL...")
# Adiciona o driver oficial do SQLite para C# apontando explicitamente para o projeto
!dotnet add VagasTechApp/VagasTechApp.csproj package Microsoft.Data.Sqlite -v 8.0.11

Criando o novo projeto de console C# 'VagasTechApp'...
=========
Welcome to .NET 8.0!
---------------------
SDK Version: 8.0.131

----------------
Installed an ASP.NET Core HTTPS development certificate.
To trust the certificate, view the instructions: https://aka.ms/dotnet-https-linux

----------------
Write your first app: https://aka.ms/dotnet-hello-world
Find out what's new: https://aka.ms/dotnet-whats-new
Explore documentation: https://aka.ms/dotnet-docs
Report issues and find source on GitHub: https://github.com/dotnet/core
Use 'dotnet --help' to see available commands or visit: https://aka.ms/dotnet-cli
--------------------------------------------------------------------------------------
The template "Console App" was created successfully.

Processing post-creation actions...
Restoring /content/VagasTechApp/VagasTechApp.csproj:
  Determining projects to restore...
  Restored /content/VagasTechApp/VagasTechApp.csproj (in 1.03 sec).
Restore succeeded.



Instalando a bib

### **1.3 Criando o arquivo físico do banco de dados**

In [ ]:
import sqlite3

try:
    print("Criando o arquivo físico do banco de dados 'vagas_tech.db'...")
    # Abre a conexão (se o arquivo não existir, o SQLite cria um arquivo .db vazio na raiz)
    conexao = sqlite3.connect('vagas_tech.db')
    print("✅ Banco 'vagas_tech.db' criado fisicamente na pasta de arquivos!")
    conexao.close()
except Exception as e:
    print(f"🚨 Erro ao criar o banco: {e}")

Criando o arquivo físico do banco de dados 'vagas_tech.db'...
✅ Banco 'vagas_tech.db' criado fisicamente na pasta de arquivos!


## **2. Desenvolvimento do CRUD**

### **2.1 Cria um novo arquivo chamado `MetodosCRUD.cs`**

In [ ]:
%%writefile VagasTechApp/MetodosCRUD.cs
using System;
using Microsoft.Data.Sqlite;

public static class MetodosCRUD
{
    public static void CadastrarVaga(
        SqliteConnection conexao,
        int idVaga,
        string titulo,
        string empresa,
        decimal salario)
    {
        if (idVaga <= 0)
            throw new ArgumentException("ID da vaga é obrigatório e deve ser maior que zero.");

        if (string.IsNullOrWhiteSpace(titulo))
            throw new ArgumentException("Título da vaga é obrigatório.");

        if (string.IsNullOrWhiteSpace(empresa))
            throw new ArgumentException("Empresa é obrigatória.");

        if (salario <= 0)
            throw new ArgumentException("Salário deve ser maior que zero.");

        string sql = @"
            INSERT INTO VAGAS (ID_VAGA, TITULO, EMPRESA, SALARIO)
            VALUES (@idVaga, @titulo, @empresa, @salario);
        ";

        try
        {
            using var comando = new SqliteCommand(sql, conexao);

            comando.Parameters.AddWithValue("@idVaga", idVaga);
            comando.Parameters.AddWithValue("@titulo", titulo.Trim());
            comando.Parameters.AddWithValue("@empresa", empresa.Trim());
            comando.Parameters.AddWithValue("@salario", salario);

            comando.ExecuteNonQuery();

            Console.WriteLine($"Vaga '{titulo}' cadastrada com sucesso!");
        }
        catch (SqliteException ex) when (ex.SqliteErrorCode == 19)
        {
            throw new InvalidOperationException($"Não foi possível cadastrar a vaga. O ID {idVaga} já existe.", ex);
        }
    }

    public static void CadastrarCandidata(
        SqliteConnection conexao,
        int idCandidata,
        string nome,
        string email)
    {
        if (idCandidata <= 0)
            throw new ArgumentException("ID da candidata é obrigatório e deve ser maior que zero.");

        if (string.IsNullOrWhiteSpace(nome))
            throw new ArgumentException("Nome da candidata é obrigatório.");

        if (string.IsNullOrWhiteSpace(email))
            throw new ArgumentException("E-mail da candidata é obrigatório.");

        string sql = @"
            INSERT INTO CANDIDATAS (ID_CANDIDATA, NOME, EMAIL)
            VALUES (@idCandidata, @nome, @email);
        ";

        try
        {
            using var comando = new SqliteCommand(sql, conexao);

            comando.Parameters.AddWithValue("@idCandidata", idCandidata);
            comando.Parameters.AddWithValue("@nome", nome.Trim());
            comando.Parameters.AddWithValue("@email", email.Trim());

            comando.ExecuteNonQuery();

            Console.WriteLine($"Candidata '{nome}' cadastrada com sucesso!");
        }
        catch (SqliteException ex) when (ex.SqliteErrorCode == 19)
        {
            throw new InvalidOperationException($"Não foi possível cadastrar a candidata. O ID {idCandidata} já existe.", ex);
        }
    }

    public static void EnviarCandidatura(
        SqliteConnection conexao,
        int idCandidatura,
        int idVaga,
        int idCandidata)
    {
        if (idCandidatura <= 0)
            throw new ArgumentException("ID da candidatura deve ser maior que zero.");

        if (idVaga <= 0)
            throw new ArgumentException("ID da vaga deve ser maior que zero.");

        if (idCandidata <= 0)
            throw new ArgumentException("ID da candidata deve ser maior que zero.");

        string consultaCandidaturaExistente = @"
            SELECT EXISTS (
                SELECT 1
                FROM CANDIDATURAS
                WHERE ID_VAGA = @idVaga
                  AND ID_CANDIDATA = @idCandidata
            );
        ";

        using var comandoCandidaturaExistente = new SqliteCommand(consultaCandidaturaExistente, conexao);

        comandoCandidaturaExistente.Parameters.AddWithValue("@idVaga", idVaga);
        comandoCandidaturaExistente.Parameters.AddWithValue("@idCandidata", idCandidata);

        bool candidaturaExiste = Convert.ToBoolean(comandoCandidaturaExistente.ExecuteScalar());

        if (candidaturaExiste)
            throw new InvalidOperationException($"A candidata {idCandidata} já possui uma candidatura para a vaga {idVaga}.");

        string sql = @"
            INSERT INTO CANDIDATURAS (ID_CANDIDATURA, DATA_ENVIO, ID_VAGA, ID_CANDIDATA)
            VALUES (@idCandidatura, @dataEnvio, @idVaga, @idCandidata);
        ";

        try
        {
            using var comando = new SqliteCommand(sql, conexao);

            comando.Parameters.AddWithValue("@idCandidatura", idCandidatura);
            comando.Parameters.AddWithValue("@dataEnvio", DateTime.Now);
            comando.Parameters.AddWithValue("@idVaga", idVaga);
            comando.Parameters.AddWithValue("@idCandidata", idCandidata);

            comando.ExecuteNonQuery();

            Console.WriteLine($"Candidatura {idCandidatura} enviada com sucesso!");
        }
        catch (SqliteException ex) when (ex.SqliteErrorCode == 19)
        {
            throw new InvalidOperationException($"Não foi possível enviar a candidatura. O ID da candidatura {idCandidatura} já existe.", ex);
        }
    }

    public static void ConsultarCandidaturas(SqliteConnection conexao)
    {
        string sql = @"
            SELECT
                C.NOME,
                C.EMAIL,
                V.TITULO,
                V.EMPRESA
            FROM CANDIDATURAS CA
            INNER JOIN CANDIDATAS C
                ON CA.ID_CANDIDATA = C.ID_CANDIDATA
            INNER JOIN VAGAS V
                ON CA.ID_VAGA = V.ID_VAGA
            ORDER BY CA.ID_CANDIDATURA;
        ";

        using var comando = new SqliteCommand(sql, conexao);
        using var leitor = comando.ExecuteReader();

        Console.WriteLine("\n========== CANDIDATURAS ==========");

        while (leitor.Read())
        {
            Console.WriteLine($"Candidata: {leitor["NOME"]}");
            Console.WriteLine($"E-mail: {leitor["EMAIL"]}");
            Console.WriteLine($"Vaga: {leitor["TITULO"]}");
            Console.WriteLine($"Empresa: {leitor["EMPRESA"]}");
            Console.WriteLine("-----------------------------------");
        }
    }

    public static void AtualizarSalarioVaga(
        SqliteConnection conexao,
        int idVaga,
        decimal novoSalario)
    {
        if (idVaga <= 0)
            throw new ArgumentException("ID da vaga deve ser maior que zero.");

        if (novoSalario <= 0)
            throw new ArgumentException("O novo salário deve ser maior que zero.");

        string query = @"
            UPDATE VAGAS
            SET SALARIO = @salario
            WHERE ID_VAGA = @id;
        ";

        using var comando = new SqliteCommand(query, conexao);

        comando.Parameters.AddWithValue("@salario", novoSalario);
        comando.Parameters.AddWithValue("@id", idVaga);

        int linhasAfetadas = comando.ExecuteNonQuery();

        if (linhasAfetadas > 0)
            Console.WriteLine($"Salário da vaga {idVaga} atualizado para R$ {novoSalario:N2}");
        else
            Console.WriteLine($"Vaga {idVaga} não encontrada.");
    }

    public static void CancelarCandidatura(
        SqliteConnection conexao,
        int idCandidatura)
    {
        if (idCandidatura <= 0)
            throw new ArgumentException("ID da candidatura deve ser maior que zero.");

        string query = @"
            DELETE FROM CANDIDATURAS
            WHERE ID_CANDIDATURA = @id;
        ";

        using var comando = new SqliteCommand(query, conexao);

        comando.Parameters.AddWithValue("@id", idCandidatura);

        int linhasAfetadas = comando.ExecuteNonQuery();

        if (linhasAfetadas > 0)
            Console.WriteLine($"Candidatura {idCandidatura} cancelada com sucesso!");
        else
            Console.WriteLine($"Candidatura {idCandidatura} não encontrada.");
    }
}

Overwriting VagasTechApp/MetodosCRUD.cs


### **2.2 Compilando o projeto para verificar se há erros de sintaxe**

In [ ]:
!dotnet build VagasTechApp

=MSBuild version 17.8.49+7806cbf7b for .NET
  Determining projects to restore...
  All projects are up-to-date for restore.
  VagasTechApp -> /content/VagasTechApp/bin/Debug/net8.0/VagasTechApp.dll

Build succeeded.
    0 Warning(s)
    0 Error(s)

Time Elapsed 00:00:10.62


## **3. Simulação e Integração**

### **3.1 Lógica de integração no arquivo `Program.cs`**

In [ ]:
%%writefile VagasTechApp/Program.cs
using System;
using Microsoft.Data.Sqlite;

namespace VagasTechApp
{
    class Program
    {
        static void Main(string[] args)
        {
            var stringConexao = "Data Source=/content/vagas_tech.db";

            try
            {
                Console.WriteLine("=================================================");
                Console.WriteLine("INICIANDO INTEGRAÇÃO DA PLATAFORMA VAGAS TECH...");
                Console.WriteLine("=================================================\n");

                using (var conexao = new SqliteConnection(stringConexao))
                {
                    conexao.Open();

                    Console.WriteLine("Conexão estabelecida com o arquivo 'vagas_tech.db'!");

                    Console.WriteLine("\n--- 1. Cadastrando vagas ---");

                    MetodosCRUD.CadastrarVaga(conexao, 1, "Engenheira de Dados", "Empresa Alfa", 9000);
                    MetodosCRUD.CadastrarVaga(conexao, 2, "Analista de BI", "Empresa Ômega", 7500);

                    Console.WriteLine("\n--- 2. Cadastrando candidata ---");

                    MetodosCRUD.CadastrarCandidata(conexao, 201, "Mariana Souza", "mariana.souza@email.com");

                    Console.WriteLine("\n--- 3. Enviando candidaturas ---");

                    MetodosCRUD.EnviarCandidatura(conexao, 901, 1, 201);
                    MetodosCRUD.EnviarCandidatura(conexao, 902, 2, 201);

                    Console.WriteLine("\n--- 4. Consulta das candidaturas ---");

                    MetodosCRUD.ConsultarCandidaturas(conexao);

                    Console.WriteLine("\n--- 5. Atualizando salário ---");

                    MetodosCRUD.AtualizarSalarioVaga(conexao, 1, 9500);

                    Console.WriteLine("\n--- 6. Cancelando candidatura ---");

                    MetodosCRUD.CancelarCandidatura(conexao, 902);

                    Console.WriteLine("\n--- 7. Consulta final das candidaturas ---");

                    MetodosCRUD.ConsultarCandidaturas(conexao);
                }
            }
            catch (Exception ex)
            {
                Console.WriteLine($"Ocorreu um erro: {ex.Message}");
            }
            finally
            {
                Console.WriteLine("\n=================================================");
                Console.WriteLine("PROCESSO DE INTEGRAÇÃO FINALIZADO.");
                Console.WriteLine("=================================================");
            }
        }
    }
}

Overwriting VagasTechApp/Program.cs
